In [1]:
# 1. Install specific libraries WITH a strict NumPy pin
!pip install -q ragas==0.1.22 datasets langchain langchain-community sentence-transformers python-dotenv langchain-huggingface "numpy<2.0.0"

# 2. Install zstd (Required by Kaggle to unzip Ollama)
!apt-get update && apt-get install -y zstd

# 3. Install the Ollama Engine onto the Kaggle Server
!curl -fsSL https://ollama.com/install.sh | sh

# 4. Start the Ollama server in the background
import subprocess
import time
print("\nStarting Ollama server...")
process = subprocess.Popen(["ollama", "serve"])
time.sleep(5) # Give the server a few seconds to boot up

# 5. Download the Llama 3.1 (8B) model to the GPU (~4.7 GB)
print("\nDownloading the Llama 3.1 model (this takes about 1-2 minutes)...")
!ollama pull llama3.1
print("\n✅ Environment fully setup and model loaded!")

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease     
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease                     
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease               
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu

time=2026-05-25T05:25:12.843Z level=INFO source=routes.go:1802 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_MULTIUSER_CACHE:false OLLAMA_NEW_ENGINE:false OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* https://0.0.0.0:* app://* fi


]11;?\[GIN] 2026/05/25 - 05:25:22 | 200 |     127.547µs |       127.0.0.1 | HEAD     "/"
pulling manifest ⠋ pulling manifest ⠙ [GIN] 2026/05/25 - 05:25:23 | 200 |  251.282012ms |       127.0.0.1 | POST     "/api/pull"
pulling manifest 
pulling 667b0c1932bc: 100% ▕██████████████████▏ 4.9 GB                         
pulling 948af2743fc7: 100% ▕██████████████████▏ 1.5 KB                         
pulling 0ba8f0e314b4: 100% ▕██████████████████▏  12 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 455f34728c9b: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 

✅ Environment fully setup and model loaded!


In [ ]:
import pandas as pd
from datasets import Dataset

# Import stable RAGAS metrics (All 4!)
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas import evaluate
from ragas.run_config import RunConfig

from langchain_community.chat_models import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings

print("1. Connecting to local GPU Llama 3.1 model...")
llm = ChatOllama(
    model="llama3.1",
    temperature=0,
    base_url="http://127.0.0.1:11434"
)

print("2. Loading Local HuggingFace Embeddings directly to the GPU...")
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

print("3. Loading raw evaluation results...")
# ⚠️ KAGGLE FILE PATH CHECK: 
# Paste your exact Kaggle file path here if it's not in the working directory!
# Example: "/kaggle/input/my-dataset/evaluation_results_raw.csv"
file_path = "/kaggle/input/datasets/ckiitp/evaluation-results-raw-csv/evaluation_results_raw.csv" 
df = pd.read_csv(file_path, encoding="cp1252")

# THE LOOP: Grading Pipelines
pipelines_to_test = ["Pipeline_A", "Pipeline_B", "Pipeline_C", "Pipeline_D"]

for pipeline in pipelines_to_test:
    print(f"\n{'='*50}")
    print(f"🚀 NOW GRADING: {pipeline} (Full 68-Question Dataset)")
    print(f"{'='*50}\n")

    ragas_data = {
        "question": df["Question"].tolist(),
        "answer": df[f"{pipeline}_Answer"].fillna("").tolist(),
        "contexts": [[str(ctx)] for ctx in df[f"{pipeline}_Context"].fillna("").tolist()],
        "ground_truth": df["Expected Answer"].fillna("").tolist()
    }
    dataset = Dataset.from_dict(ragas_data)

    try:
        result = evaluate(
            dataset=dataset,
            metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
            llm=llm,
            embeddings=embeddings,
            run_config=RunConfig(max_workers=1, max_retries=3),
            raise_exceptions=False
        )

        graded_df = result.to_pandas()
        
        # Save output to Kaggle's working directory so you can download it
        output_path = f"/kaggle/working/{pipeline}_graded_results.csv"
        graded_df.to_csv(output_path, index=False)

        print(f"\n✅ {pipeline} Grading Complete!")
        print(result)

    except Exception as e:
        print(f"\n❌ {pipeline} CRASHED: {repr(e)}")

print("\n🎉 ALL PIPELINES FULLY EVALUATED! DOWNLOAD YOUR CSV FILES!")

: 